# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a reproducible guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is accessible via a Croissant schema URL (see below).

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata (as a Dataset object)
dataset = mlc.Dataset(croissant_url)
# The metadata object (not a dictionary!)
md = dataset.metadata

print(f"Dataset Name: {md.name}")
print(f"Dataset Description: {md.description}")

## 2. Data Overview
List available record sets and their fields by their `@id`. This overview helps us discover top-level data tables and their internal schema.

**Note:** `mlcroissant` refers to entities like record sets, fields, and columns using their `@id`. We'll use those identifiers throughout for clarity and reproducibility.

In [ ]:
# List all available record sets and their field ids
print("Available record sets and fields:")

record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id} (name: {record_set.name})")
    field_ids = [field.id for field in record_set.fields]
    print(f"  Fields (@id): {field_ids}")
    record_sets.append(record_set.id)

if not record_sets:
    print("No record sets were found in the dataset metadata.\nIf this is unexpected, ensure the Croissant schema at the given URL exposes record sets.")

## 3. Data Extraction
Load data from each record set into pandas DataFrames using the `@id` values from the overview.

If multiple record sets exist, loop across all. If none are available, demonstrate with error handling.

In [ ]:
# Extract records from each record set defined by `@id`. Store DataFrames by id.
dfs = {}

if record_sets:
    for rs_id in record_sets:
        # Use mlcroissant to get all records (dicts) for this record set
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dfs[rs_id] = df
            print(f"Loaded DataFrame for record set '{rs_id}' with shape {df.shape}")
        except Exception as e:
            print(f"Could not load records for record set {rs_id}: {e}")
    if dfs:
        # Display DataFrame columns for first available record set
        first_rs_id = list(dfs.keys())[0]
        print(f"\nColumns in DataFrame for record set '@id': {first_rs_id}")
        print(dfs[first_rs_id].columns.tolist())
        display(dfs[first_rs_id].head())
    else:
        print("No records or data could be loaded from the record sets.")
else:
    print("No record sets found in the dataset; skipping data extraction.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from one of the record sets (by `@id`) and demonstrate basic analysis steps such as filtering, normalization, and grouping. We reference all columns and fields by their `@id`.

If no record sets or data are loaded, this section illustrates general code structure and error handling.

In [ ]:
# Pick a record set and numeric field (by @id) for demonstration
import numpy as np
example_rs_id = None
numeric_field_id = None
group_field_id = None

if dfs:
    # Attempt to auto-detect a numeric field
    for rs_id, df in dfs.items():
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                example_rs_id = rs_id
                numeric_field_id = col
                break
        if numeric_field_id:
            break

    if example_rs_id and numeric_field_id:
        print(f"Selected record set: {example_rs_id}, numeric field: {numeric_field_id}")
        threshold = np.nanmean(dfs[example_rs_id][numeric_field_id])
        print(f"Filtering {numeric_field_id} > {threshold:.2f}")
        filtered_df = dfs[example_rs_id][dfs[example_rs_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the numeric field
        mu = filtered_df[numeric_field_id].mean()
        sigma = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mu) / sigma
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try to group by a string/categorical field (first object dtype column not the numeric one)
        for col in dfs[example_rs_id].columns:
            if col != numeric_field_id and dfs[example_rs_id][col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group (categorical) field found for grouping.")
    else:
        print("No numeric field detected in any loaded DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field (referenced by its `@id`) and, if possible, its mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and example_rs_id and numeric_field_id:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(dfs[example_rs_id][numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{example_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in dfs[example_rs_id].columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dfs[example_rs_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated programmatic exploration of the FAIR^2 dataset using `mlcroissant`:

- Loaded metadata and record sets from the Croissant schema using their `@id` fields for reproducibility.
- Presented table and field overviews, and loaded data into DataFrames.
- Showed basic EDA (filtering, normalization, grouping) and visualized results.
- All dataset entities (record sets, fields) were referenced using `@id` to ensure precise correspondence to the schema.

This workflow supports transparent, reproducible data science on complex, schema-driven datasets.